In [1]:
import pandas as pd
import chardet
import logging
from pathlib import Path
from datetime import date

# I use logging instead of print so output is properly timestamped and levelled.
# In production on Databricks this integrates directly with the cluster log driver.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

# Paths are defined relative to the notebook so they work on any machine.
BASE_DIR      = Path("..")
RAW_DATA_PATH = BASE_DIR / "data" / "raw" / "RawData.csv"
COLUMNS_PATH  = BASE_DIR / "data" / "raw" / "Columns.csv"

# I follow the Raw, Prepared, Staging naming convention for each layer.
RAW_DIR      = BASE_DIR / "data" / "raw_layer"
PREPARED_DIR = BASE_DIR / "data" / "prepared_layer"
STAGING_DIR  = BASE_DIR / "data" / "staging_layer"

for d in [RAW_DIR, PREPARED_DIR, STAGING_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# In production I would receive this as a parameter from Azure Data Factory.
INGESTION_DATE = date.today().isoformat()

logger.info("Ingestion date : %s", INGESTION_DATE)
logger.info("Raw data path  : %s", RAW_DATA_PATH.resolve())

2026-03-09 18:15:56  INFO  Ingestion date : 2026-03-09


2026-03-09 18:15:56  INFO  Raw data path  : /home/capinha/python_projects/data_engineering_sandbox/vasco_business_case/data/raw/RawData.csv


In [2]:
# I read 200KB of the file as raw bytes before decoding anything.
# Sampling is a common practice: chardet's confidence stabilises well within the first
# few KB so reading the whole file would not change the result but would be slower
# on larger files in production.
with open(RAW_DATA_PATH, "rb") as f:
    raw_bytes = f.read(200_000)

# chardet looks for byte frequency patterns to figure out the encoding.
detection         = chardet.detect(raw_bytes)
detected_encoding = detection["encoding"]
confidence        = detection["confidence"]

logger.info("Detected encoding : %s (confidence %.1f%%)", detected_encoding, confidence * 100)

# Below 70% confidence I do not trust the result and fall back to cp1252.
# cp1252 is Windows ANSI and the most common encoding for survey tool exports.
if confidence < 0.7:
    detected_encoding = "cp1252"
    logger.warning("Low confidence detected, falling back to %s", detected_encoding)

# If encoding issues keep showing up, it is worth raising with the business.
# In a past project I dealt with a cp1252 file that had characters outside the set.
# We tried regex cleaning but the compute cost was too high so we ended up requesting
# a fresh export, which was faster and cheaper than trying to fix corrupted characters.
# For now I proceed with whatever encoding was detected since some data is better than none.

2026-03-09 18:15:57  INFO  Detected encoding : utf-8 (confidence 80.4%)


In [3]:
# low_memory=False makes pandas read the full column before inferring its type.
# Without it, mixed-type columns on large files can trigger warnings.
df_raw = pd.read_csv(
    RAW_DATA_PATH,
    encoding=detected_encoding,
    low_memory=False,
)

# I tag each row so any record can always be traced back to the file and run that created it.
df_raw["_ingestion_date"] = INGESTION_DATE
df_raw["_source_file"]    = RAW_DATA_PATH.name

logger.info("Rows loaded : %s", f"{len(df_raw):,}")
logger.info("Columns     : %s", df_raw.shape[1])

# Writing to Parquet is where the encoding fix happens automatically.
# Parquet stores all strings as UTF-8 internally so every downstream layer
# reads clean UTF-8 regardless of what the source encoding was.
raw_path = RAW_DIR / f"survey_raw_{INGESTION_DATE}.parquet"
df_raw.to_parquet(raw_path, index=False)

logger.info("Raw layer written : %s (%.1f KB)", raw_path, raw_path.stat().st_size / 1024)

2026-03-09 18:15:57  INFO  Rows loaded : 10,000


2026-03-09 18:15:57  INFO  Columns     : 131


2026-03-09 18:15:58  INFO  Raw layer written : ../data/raw_layer/survey_raw_2026-03-09.parquet (1170.2 KB)


In [4]:
# I work from a copy so the raw layer dataframe is never touched by any transformation.
df_prepared = df_raw.copy()

logger.info("Starting row count : %s", f"{len(df_prepared):,}")

# Business Rule 1: respondents must be unique, keeping the first occurrence.
# Duplicates can show up if the same file is processed twice, if the source exports
# overlapping date ranges, or if the survey platform is not configured to prevent
# someone from submitting more than once. The business should be notified if any
# are found so they can look into the root cause.
dupes       = df_prepared["Respondent"].duplicated().sum()
df_prepared = df_prepared.drop_duplicates(subset="Respondent", keep="first")

logger.info("Duplicate respondents removed : %s", f"{dupes:,}")
logger.info("Rows after deduplication      : %s", f"{len(df_prepared):,}")

2026-03-09 18:15:58  INFO  Starting row count : 10,000


2026-03-09 18:15:58  INFO  Duplicate respondents removed : 0


2026-03-09 18:15:58  INFO  Rows after deduplication      : 10,000


In [5]:
# Business Rule 2: blank Student means No.
# I fill rather than drop because a blank here is a known business default, not missing data.
student_nulls            = df_prepared["Student"].isna().sum()
df_prepared["Student"]   = df_prepared["Student"].fillna("No")

logger.info("Student nulls defaulted to No                     : %s", f"{student_nulls:,}")

# Business Rule 3: blank Employment means full-time. Same reasoning as above.
employment_nulls             = df_prepared["Employment"].isna().sum()
df_prepared["Employment"]    = df_prepared["Employment"].fillna("Employed full-time")

logger.info("Employment nulls defaulted to Employed full-time  : %s", f"{employment_nulls:,}")

2026-03-09 18:15:58  INFO  Student nulls defaulted to No                     : 232


2026-03-09 18:15:58  INFO  Employment nulls defaulted to Employed full-time  : 189


In [6]:
# Business Rule 4: responses with more than 3 empty fields are considered invalid.
# Instead of dropping rows straight away I add flag columns so all data is kept
# in the Prepared layer. The business can then filter, audit, or adjust the threshold
# without losing anything. If they confirm the rule, dropping is a single filter away.

# Pipeline metadata columns are excluded since they are not survey fields.
survey_cols = [c for c in df_prepared.columns if not c.startswith("_")]

# empty_fields is a list of the column names that are null for each row.
# If the same fields keep appearing across many rows it may point to a survey
# design problem or a question that respondents consistently skip.
df_prepared["empty_fields"] = df_prepared[survey_cols].apply(
    lambda row: [col for col in survey_cols if pd.isna(row[col])], axis=1
)

# Storing the count separately makes threshold comparisons easy and gives more
# context than a boolean alone. A row with 4 nulls is very different from one with 80.
df_prepared["empty_field_count"] = df_prepared["empty_fields"].apply(len)

# This boolean is the direct expression of the business rule, derived from the count
# so all three columns always stay consistent with each other.
df_prepared["more_than_three_empty"] = df_prepared["empty_field_count"] > 3

flagged = df_prepared["more_than_three_empty"].sum()
logger.info("Rows flagged as more than three empty : %s of %s (%.1f%%)",
            f"{flagged:,}", f"{len(df_prepared):,}", flagged / len(df_prepared) * 100)
logger.info("Rows that pass the rule               : %s",
            f"{(~df_prepared['more_than_three_empty']).sum():,}")

2026-03-09 18:16:00  INFO  Rows flagged as more than three empty : 9,603 of 10,000 (96.0%)


2026-03-09 18:16:00  INFO  Rows that pass the rule               : 397


In [7]:
# Business Rule 5: all salaries should be yearly.
# I create salary_yearly as a new column and keep the original Salary and SalaryType
# untouched so the raw values are always there for audit if needed.
# For cross-currency comparisons, ConvertedSalary already exists in the dataset.
# The survey provider computed it at 2018-01-18 USD exchange rates and it is already
# annualised, so no further conversion is needed for that purpose.

def normalize_salary(row):
    salary = row["Salary"]
    stype  = row["SalaryType"]
    if pd.isna(salary):
        return None
    if stype == "Weekly":
        return salary * 52
    if stype == "Monthly":
        return salary * 12
    # If SalaryType is Yearly or null but a salary value exists, I treat it as yearly.
    return salary

df_prepared["salary_yearly"] = df_prepared.apply(normalize_salary, axis=1)

logger.info("Rows with a yearly salary value : %s", f"{df_prepared['salary_yearly'].notna().sum():,}")
logger.info("Rows without salary data        : %s", f"{df_prepared['salary_yearly'].isna().sum():,}")

2026-03-09 18:16:00  INFO  Rows with a yearly salary value : 7,406


2026-03-09 18:16:00  INFO  Rows without salary data        : 2,594


In [8]:
# All business rules are applied. I write the Prepared layer now.
# Downstream consumers read from here rather than from the raw CSV.
prepared_path = PREPARED_DIR / f"survey_prepared_{INGESTION_DATE}.parquet"
df_prepared.to_parquet(prepared_path, index=False)

logger.info("Prepared layer written : %s (%.1f KB)", prepared_path, prepared_path.stat().st_size / 1024)
logger.info("Row count              : %s", f"{len(df_prepared):,}")
logger.info("Column count           : %s", df_prepared.shape[1])

2026-03-09 18:16:00  INFO  Prepared layer written : ../data/prepared_layer/survey_prepared_2026-03-09.parquet (1323.4 KB)


2026-03-09 18:16:00  INFO  Row count              : 10,000


2026-03-09 18:16:00  INFO  Column count           : 135


In [9]:
# The Staging layer contains only what the business asked for.
# Analysis 1: which survey questions have the highest variance in answers across participants.
# Analysis 2: whether there is a relationship between years of coding experience and salary.
# I also kept key demographic columns so both analyses can be broken down by participant
# profile if needed. The full 129-column dataset stays in the Prepared layer.

RENAME_MAP = {
    "Respondent"         : "respondent_id",
    "Country"            : "country",
    "Student"            : "student",
    "Employment"         : "employment",
    "FormalEducation"    : "formal_education",
    "UndergradMajor"     : "undergrad_major",
    "CompanySize"        : "company_size",
    "DevType"            : "dev_type",
    "YearsCoding"        : "years_coding",
    "YearsCodingProf"    : "years_coding_prof",
    "JobSatisfaction"    : "job_satisfaction",
    "CareerSatisfaction" : "career_satisfaction",
    "Salary"             : "salary_original",
    "SalaryType"         : "salary_type",
    "ConvertedSalary"    : "converted_salary_usd",
    "Currency"           : "currency",
    "CurrencySymbol"     : "currency_symbol",
    "Gender"             : "gender",
    "Age"                : "age",
}

df_staging = df_prepared.rename(columns=RENAME_MAP)

# The flag columns from Business Rule 4 are included so analysts can filter them out
# once the business confirms whether to keep or discard those rows.
STAGING_COLS = [
    "respondent_id", "country", "student", "employment",
    "formal_education", "undergrad_major", "company_size", "dev_type",
    "years_coding", "years_coding_prof",
    "job_satisfaction", "career_satisfaction",
    "salary_original", "salary_type", "salary_yearly", "converted_salary_usd",
    "currency", "currency_symbol",
    "gender", "age",
    "empty_field_count", "more_than_three_empty",
    "_ingestion_date", "_source_file",
]

df_staging = df_staging[[c for c in STAGING_COLS if c in df_staging.columns]]

staging_path = STAGING_DIR / f"fact_survey_{INGESTION_DATE}.parquet"
df_staging.to_parquet(staging_path, index=False)

logger.info("Staging layer written : %s (%.1f KB)", staging_path, staging_path.stat().st_size / 1024)
logger.info("Rows %s  Columns %s", f"{len(df_staging):,}", df_staging.shape[1])

2026-03-09 18:16:00  INFO  Staging layer written : ../data/staging_layer/fact_survey_2026-03-09.parquet (279.5 KB)


2026-03-09 18:16:00  INFO  Rows 10,000  Columns 24


In [10]:
# Analysis 1: variance per survey question.
# df_prepared is still in memory so no extra read from disk is needed.

# I grab all numeric columns to start with.
numeric_cols = df_prepared.select_dtypes(include="number").columns.tolist()

# Pipeline and flag columns are not survey questions so I remove them.
exclude      = {"empty_field_count", "more_than_three_empty"}
question_cols = [c for c in numeric_cols if c not in exclude]

# I compute variance across all respondents and rank from highest to lowest.
variance_df = (
    df_prepared[question_cols]
    .var()
    .reset_index()
    .rename(columns={"index": "question", 0: "variance"})
    .sort_values("variance", ascending=False)
    .reset_index(drop=True)
)

variance_path = STAGING_DIR / f"fact_variance_{INGESTION_DATE}.parquet"
variance_df.to_parquet(variance_path, index=False)

logger.info("Variance table written : %s", variance_path)
logger.info("Top 10 questions by variance:")
for _, row in variance_df.head(10).iterrows():
    logger.info("  %-30s  %.2f", row["question"], row["variance"])

2026-03-09 18:16:00  INFO  Variance table written : ../data/staging_layer/fact_variance_2026-03-09.parquet


2026-03-09 18:16:00  INFO  Top 10 questions by variance:


2026-03-09 18:16:00  INFO    ConvertedSalary                 40885354284.07


2026-03-09 18:16:00  INFO    Respondent                      863848800.53


2026-03-09 18:16:00  INFO    AssessBenefits2                 9.35


...

2026-03-09 18:16:00  INFO    AssessBenefits3                 7.97


2026-03-09 18:16:00  INFO    AssessBenefits4                 7.85


2026-03-09 18:16:00  INFO    AssessBenefits8                 7.73


2026-03-09 18:16:00  INFO    AssessJob1                      7.72


2026-03-09 18:16:00  INFO    AssessBenefits9                 7.58


In [11]:
# The output above shows ConvertedSalary and Respondent at the top with variance in the billions.
# That is expected since they are a continuous monetary value and a row identifier,
# not bounded survey ranking questions. Keeping them makes the ranking meaningless
# so I remove them along with salary_yearly which has the same problem.
# What stays are the actual survey ranking questions like AssessJob and AssessBenefits
# where variance genuinely reflects disagreement between respondents.
# Once the business confirms this is the right approach, these columns can be removed
# from the initial calculation above to save compute on every pipeline run.

non_question_cols   = {"ConvertedSalary", "Respondent", "salary_yearly"}
question_cols_clean = [c for c in question_cols if c not in non_question_cols]

variance_df_clean = (
    df_prepared[question_cols_clean]
    .var()
    .reset_index()
    .rename(columns={"index": "question", 0: "variance"})
    .sort_values("variance", ascending=False)
    .reset_index(drop=True)
)

# I overwrite the variance file with this cleaned version.
variance_df_clean.to_parquet(variance_path, index=False)

logger.info("Top 10 survey questions by variance after removing non-question columns:")
for _, row in variance_df_clean.head(10).iterrows():
    logger.info("  %-30s  %.2f", row["question"], row["variance"])

2026-03-09 18:16:00  INFO  Top 10 survey questions by variance after removing non-question columns:


2026-03-09 18:16:00  INFO    AssessBenefits2                 9.35


2026-03-09 18:16:00  INFO    AssessJob7                      8.39


2026-03-09 18:16:00  INFO    AssessBenefits6                 8.00


2026-03-09 18:16:00  INFO    AssessBenefits3                 7.97


...

2026-03-09 18:16:00  INFO    AssessBenefits8                 7.73


2026-03-09 18:16:00  INFO    AssessJob1                      7.72


2026-03-09 18:16:00  INFO    AssessBenefits9                 7.58


2026-03-09 18:16:00  INFO    AssessBenefits7                 7.39


2026-03-09 18:16:00  INFO    AssessBenefits11                7.21


In [ ]:
# Analysis 2: relationship between years of coding experience and salary.

# YearsCoding is stored as a text range like "6-8 years" so I map each range
# to its numeric midpoint. The midpoint is the simplest fair representation.
YEARS_MAP = {
    "0-2 years"        : 1,
    "3-5 years"        : 4,
    "6-8 years"        : 7,
    "9-11 years"       : 10,
    "12-14 years"      : 13,
    "15-17 years"      : 16,
    "18-20 years"      : 19,
    "21-23 years"      : 22,
    "24-26 years"      : 25,
    "27-29 years"      : 28,
    "30 or more years" : 32,
}

df_prepared["years_coding_numeric"] = df_prepared["YearsCoding"].map(YEARS_MAP)

# I use ConvertedSalary rather than salary_yearly because it normalises all currencies
# to USD at a fixed 2018 rate. salary_yearly keeps local currency values which would
# skew the correlation (a salary in Indian rupees versus US dollars, for example).
salary_years = df_prepared[["years_coding_numeric", "ConvertedSalary"]].dropna()

# Pearson correlation: 1 is a perfect positive relationship, -1 is perfect negative,
# 0 means no linear relationship at all.
correlation = salary_years["years_coding_numeric"].corr(salary_years["ConvertedSalary"])

logger.info("Rows used for correlation                    : %s", f"{len(salary_years):,}")
logger.info("Correlation (YearsCoding vs ConvertedSalary) : %.4f", correlation)
logger.info("%s",
    "positive relationship, more experience tends to mean higher salary" if correlation > 0
    else "negative relationship, more experience tends to mean lower salary")

2026-03-09 18:16:00  INFO  Rows used for correlation                    : 7,244


2026-03-09 18:16:00  INFO  Correlation (YearsCoding vs ConvertedSalary) : 0.1536


2026-03-09 18:16:00  INFO  Interpretation : positive relationship, more experience tends to mean higher salary
